# Whisper's transcription plus Pyannote's Diarization 

Andrej Karpathy [suggested](https://twitter.com/karpathy/status/1574476200801538048?s=20&t=s5IMMXOYjBI6-91dib6w8g) training a classifier on top of	OpenAI [Whisper](https://openai.com/blog/whisper/) model features to identify the speaker, so we can visualize the speaker in the transcript. But, as [pointed out](https://twitter.com/tarantulae/status/1574493613362388992?s=20&t=s5IMMXOYjBI6-91dib6w8g) by Christian Perone, it seems that features from whisper wouldn't be that great for speaker recognition as its main objective is basically to ignore speaker differences.

In the following, I use [**`pyannote-audio`**](https://github.com/pyannote/pyannote-audio), a speaker diarization toolkit by Hervé Bredin, to identify the speakers, and then match it with the transcriptions of Whispr. I do it on the first 30 minutes of	Lex's 2nd [interview](https://youtu.be/SGzMElJ11Cc) with Yann LeCun. Check the result [**here**](https://majdoddin.github.io/lexicap.html). 

It is tricky to match the transcriptions to diarization segemtns, specially when the speaker changes. To resolve it, Sarah Kaiser [suggested](https://github.com/openai/whisper/discussions/264#discussioncomment-3825375) runnnig the pyannote.audio first and	then just running whisper on the split-by-speaker chunks. 
For sake of performance (and transcription quality?), we attach the audio segements into a single audio file with a silent spacer as a seperator, and run whisper on it. Enjoy it!

# Preparing the audio file

 Installing `yt-dlp` and downloading the [video](https://).

In [15]:
!pip install -U yt-dlp


[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip


In [16]:
!wget -O - -q	https://github.com/yt-dlp/FFmpeg-Builds/releases/download/latest/ffmpeg-master-latest-linux64-gpl.tar.xz | xz -qdc| tar -x

In [4]:
!yt-dlp -xv --ffmpeg-location ffmpeg-master-latest-linux64-gpl/bin --audio-format wav	-o fkxVtGHE6V4.wav -- https://youtu.be/fkxVtGHE6V4

[debug] Command-line config: ['-xv', '--ffmpeg-location', 'ffmpeg-master-latest-linux64-gpl/bin', '--audio-format', 'wav', '-o', 'fkxVtGHE6V4.wav', '--', 'https://youtu.be/fkxVtGHE6V4']
[debug] Encodings: locale UTF-8, fs utf-8, pref UTF-8, out utf-8, error utf-8, screen utf-8
[debug] yt-dlp version stable@2025.02.19 from yt-dlp/yt-dlp [4985a4041] (pip)
[debug] Python 3.11.2 (CPython x86_64 64bit) - Linux-6.8.0-55-generic-x86_64-with-glibc2.39 (OpenSSL 3.0.13 30 Jan 2024, glibc 2.39)
[debug] exe versions: ffmpeg N-118860-g81c50c33b6-20250318 (setts), ffprobe N-118860-g81c50c33b6-20250318
[debug] Optional libraries: certifi-2023.07.22, requests-2.31.0, sqlite3-3.45.1, urllib3-1.26.19
[debug] Proxy map: {}
[debug] Request Handlers: urllib
[debug] Loaded 1841 extractors
[youtube] Extracting URL: https://youtu.be/fkxVtGHE6V4
[youtube] fkxVtGHE6V4: Downloading webpage
[youtube] fkxVtGHE6V4: Downloading tv client config
[youtube] fkxVtGHE6V4: Downloading player 7d1d50a6
[youtube] fkxVtGHE6V4


Cutting the first 20 minutes of the video for further process.


In [18]:
!pip install pydub


[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip


In [19]:
from pydub import AudioSegment

t1 = 0 * 1000 #Works in milliseconds
t2 = 20 * 60 * 1000

newAudio = AudioSegment.from_wav("lecun.wav")
a = newAudio[t1:t2]
a.export("lecun1.wav", format="wav") 


<_io.BufferedRandom name='lecun1.wav'>

`pyannote.audio` seems to miss the first 0.5 seconds of the audio, and, therefore, we prepend a spcacer.

In [20]:
audio = AudioSegment.from_wav("lecun1.wav")
spacermilli = 2000
spacer = AudioSegment.silent(duration=spacermilli)
audio = spacer.append(audio, crossfade=0)

audio.export('audio.wav', format='wav')

<_io.BufferedRandom name='audio.wav'>

# Pyannote's Diarization

[`pyannote.audio`](https://github.com/pyannote/pyannote-audio) is an open-source toolkit written in Python for **speaker diarization**. 

Based on [`PyTorch`](https://pytorch.org) machine learning framework, it provides a set of trainable end-to-end neural building blocks that can be combined and jointly optimized to build speaker diarization pipelines. 

`pyannote.audio` also comes with pretrained [models](https://huggingface.co/models?other=pyannote-audio-model) and [pipelines](https://huggingface.co/models?other=pyannote-audio-pipeline) covering a wide range of domains for voice activity detection, speaker segmentation, overlapped speech detection, speaker embedding reaching state-of-the-art performance for most of them. 

Installing Pyannote and running it on the video to generate the diarizations.

In [21]:
!pip install	 pyannote.audio


[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip


In [66]:
from pyannote.audio import Pipeline
import os
import torch

pipeline = Pipeline.from_pretrained('pyannote/speaker-diarization-3.1', use_auth_token=os.environ['HF_TOKEN'])
pipeline.to(torch.device("cuda"))


In [67]:
DEMO_FILE = {'uri': 'blabal', 'audio': 'audio.wav'}
dz = pipeline(DEMO_FILE)	

with open("diarization.txt", "w") as text_file:
		text_file.write(str(dz))

/home/foster/.pyenv/versions/3.11.2/lib/python3.11/site-packages/pyannote/audio/models/blocks/pooling.py:104: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1831.)
  std = sequences.std(dim=-1, correction=1)


In [68]:
print(*list(dz.itertracks(yield_label = True))[:10], sep="\n")

(<Segment(2.00534, 13.8178)>, 'A', 'SPEAKER_00')
(<Segment(15.9947, 27.0478)>, 'B', 'SPEAKER_00')
(<Segment(27.3347, 31.0135)>, 'C', 'SPEAKER_00')
(<Segment(31.0303, 31.0641)>, 'D', 'SPEAKER_00')
(<Segment(31.0641, 32.431)>, 'E', 'SPEAKER_01')
(<Segment(32.566, 42.6572)>, 'F', 'SPEAKER_00')
(<Segment(41.206, 41.5435)>, 'G', 'SPEAKER_01')
(<Segment(42.7247, 43.8891)>, 'H', 'SPEAKER_00')
(<Segment(42.826, 43.2647)>, 'I', 'SPEAKER_01')
(<Segment(44.1591, 60.4603)>, 'J', 'SPEAKER_01')


In [7]:
def millisec(timeStr):
	spl = timeStr.split(":")
	s = (int)((int(spl[0]) * 60 * 60 + int(spl[1]) * 60 + float(spl[2]) )* 1000)
	return s

In [70]:
import re
dz = open('diarization.txt').read().splitlines()
dzList = []
for l in dz:
	start, end =	tuple(re.findall('[0-9]+:[0-9]+:[0-9]+\.[0-9]+', string=l))
	start = millisec(start) - spacermilli
	end = millisec(end)	- spacermilli
	lex = not re.findall('SPEAKER_01', string=l)
	dzList.append([start, end, lex])

print(*dzList[:10], sep='\n')

[5, 11817, True]
[13994, 25047, True]
[25334, 29013, True]
[29030, 29064, True]
[29064, 30430, False]
[30564, 40657, True]
[39205, 39543, False]
[40724, 41889, True]
[40825, 41264, False]
[42159, 58460, False]


# Preparing audio file from the diarization

Attaching audio segements according to the diarization, with a spacer as the delimiter.

In [71]:
from pydub import AudioSegment
import re 

sounds = spacer
segments = []

dz = open('diarization.txt').read().splitlines()
for l in dz:
	start, end =	tuple(re.findall('[0-9]+:[0-9]+:[0-9]+\.[0-9]+', string=l))
	start = int(millisec(start)) #milliseconds
	end = int(millisec(end))	#milliseconds
	
	segments.append(len(sounds))
	sounds = sounds.append(audio[start:end], crossfade=0)
	sounds = sounds.append(spacer, crossfade=0)

sounds.export("dz.wav", format="wav") #Exports to a wav file in the current path.

<_io.BufferedRandom name='dz.wav'>

In [72]:
segments[:8]

[2000, 15812, 28865, 34544, 36578, 39944, 52036, 54374]

Freeing up some memory

In [73]:
del	 sounds, DEMO_FILE, pipeline, spacer,	audio, dz, a, newAudio

# Whisper's Transcriptions

Installing Open AI whisper.

**Important:** There is a version conflict with pyannote.audio resulting in an error (see this RP). Our workaround is to first run Pyannote and then whisper. You can safely ignore the error.


In [74]:
!pip install git+https://github.com/openai/whisper.git 

  Cloning https://github.com/openai/whisper.git to /tmp/pip-req-build-_7d348zt
  Running command git clone --filter=blob:none --quiet https://github.com/openai/whisper.git /tmp/pip-req-build-_7d348zt
  Resolved https://github.com/openai/whisper.git to commit 517a43ecd132a2089d85f4ebc044728a71d49f6e
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done

[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip


Running Open AI whisper on the prepared audio file. [link text](https://) It writes the transcription into a file.

In [5]:
!whisper fkxVtGHE6V4.wav --language es --model medium --device cuda

[00:00.000 --> 00:05.680]  Lo que parecía ser el mejor regalo de 15 años se convirtió en un trágico accidente.
[00:05.680 --> 00:10.800]  Ella es Karol Bastidas, una joven caleña que perdió sus brazos pero no la templanza con
[00:10.800 --> 00:14.200]  la que enfrenta el mundo.
[00:14.200 --> 00:19.800]  Cuando las cosas están destinadas a pasar, pasan y yo soy fiel creyente de que todo está
[00:19.800 --> 00:25.600]  escrito y de que si Dios tiene algo en el camino para uno, pues se da o se da.
[00:25.600 --> 00:31.280]  Y este es el caso de Karol que mejor dicho no sé cómo presentarte porque siento que
[00:31.280 --> 00:37.040]  esta historia tuya es muy mágica pero además muy especial porque es un caso distinto.
[00:37.040 --> 00:41.920]  Ya ustedes saben que nosotros tenemos una convocatoria abierta y que todas las mujeres
[00:41.920 --> 00:46.800]  que quieren hacer parte de estos episodios se suscriben y hacemos un proceso de curaduría
[00:46.800 --> 00:48.520]  pero con Karol fu

Reading the transcription file.

In [27]:
!pip install -U webvtt-py


[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip


In [8]:
import webvtt

captions = [[(int)(millisec(caption.start)), (int)(millisec(caption.end)),	caption.text] for caption in webvtt.read('fkxVtGHE6V4.vtt')]
print(*captions[:8], sep='\n')

[0, 5680, 'Lo que parecía ser el mejor regalo de 15 años se convirtió en un trágico accidente.']
[5680, 10800, 'Ella es Karol Bastidas, una joven caleña que perdió sus brazos pero no la templanza con']
[10800, 14200, 'la que enfrenta el mundo.']
[14200, 19800, 'Cuando las cosas están destinadas a pasar, pasan y yo soy fiel creyente de que todo está']
[19800, 25600, 'escrito y de que si Dios tiene algo en el camino para uno, pues se da o se da.']
[25600, 31280, 'Y este es el caso de Karol que mejor dicho no sé cómo presentarte porque siento que']
[31280, 37040, 'esta historia tuya es muy mágica pero además muy especial porque es un caso distinto.']
[37040, 41920, 'Ya ustedes saben que nosotros tenemos una convocatoria abierta y que todas las mujeres']


# Matching the Transcriptions and the Diarizations

Matching each trainscrition line to some diarizations, and generating the HTML file. To get the correct timing, we should take care of the parts in original audio that were in no diarization segment.

In [9]:
import yt_dlp
ydl = yt_dlp.YoutubeDL()

video_id = 'fkxVtGHE6V4'
url = 'https://youtu.be/'+video_id
info = ydl.extract_info(url, download=False)

title = info['title']
preS = '<!DOCTYPE html>\n<html lang="en">\n	<head>\n		<meta charset="UTF-8">\n		<meta name="viewport" content="width=device-width, initial-scale=1.0">\n		<meta http-equiv="X-UA-Compatible" content="ie=edge">\n		<title>Lexicap</title>\n		<style>\n				body {\n						font-family: sans-serif;\n						font-size: 18px;\n						color: #111;\n						padding: 0 0 1em 0;\n				}\n				.l {\n					color: #050;\n				}\n				.s {\n						display: inline-block;\n				}\n				.e {\n						display: inline-block;\n				}\n				.t {\n						display: inline-block;\n				}\n				#player {\n\t\tposition: sticky;\n\t\ttop: 20px;\n\t\tfloat: right;\n\t}\n		</style>\n	</head>\n	<body>\n		<h2>''' + title + ''' </h2>\n	<div	id="player"></div>\n		<script>\n			var tag = document.createElement(\'script\');\n			tag.src = "https://www.youtube.com/iframe_api";\n			var firstScriptTag = document.getElementsByTagName(\'script\')[0];\n			firstScriptTag.parentNode.insertBefore(tag, firstScriptTag);\n			var player;\n			function onYouTubeIframeAPIReady() {\n				player = new YT.Player(\'player\', {\n					height: \'210\',\n					width: \'340\',\n					videoId: '''+"'"+video_id+"'"+ ''',\n				});\n			}\n			function setCurrentTime(timepoint) {\n				player.seekTo(timepoint);\n	 player.playVideo();\n	 }\n		</script><br>\n'''
postS = '\t</body>\n</html>'

[youtube] Extracting URL: https://youtu.be/fkxVtGHE6V4
[youtube] fkxVtGHE6V4: Downloading webpage
[youtube] fkxVtGHE6V4: Downloading tv client config
[youtube] fkxVtGHE6V4: Downloading player 7d1d50a6
[youtube] fkxVtGHE6V4: Downloading tv player API JSON
[youtube] fkxVtGHE6V4: Downloading ios player API JSON
[youtube] fkxVtGHE6V4: Downloading m3u8 information


In [10]:
from datetime import timedelta

html = list(preS)

for idx in range(len(captions)):

		c = captions[idx]	
		print(c)
		start =  c[0] + 800

		if start < 0: 
			start = 0

		start = start / 1000.0
		startStr = '{0:02d}:{1:02d}:{2:02.2f}'.format((int)(start // 3600), 
																						(int)(start % 3600 // 60), 
																						start % 60)
		
		html.append('\t\t\t<div class="c">\n')
		html.append(f'\t\t\t\t<a class="l" href="#{startStr}" id="{startStr}">link</a> |\n')
		html.append(f'\t\t\t\t<div class="s"><a href="javascript:void(0);" onclick=setCurrentTime({int(start)})>{startStr}</a></div>\n')
		html.append(f'\t\t\t\t<div class="t">{c[2]}</div>\n')
		html.append('\t\t\t</div>\n\n')

html.append(postS)
s = "".join(html)

with open("lexicap.html", "w") as text_file:
		text_file.write(s)


[0, 5680, 'Lo que parecía ser el mejor regalo de 15 años se convirtió en un trágico accidente.']
[5680, 10800, 'Ella es Karol Bastidas, una joven caleña que perdió sus brazos pero no la templanza con']
[10800, 14200, 'la que enfrenta el mundo.']
[14200, 19800, 'Cuando las cosas están destinadas a pasar, pasan y yo soy fiel creyente de que todo está']
[19800, 25600, 'escrito y de que si Dios tiene algo en el camino para uno, pues se da o se da.']
[25600, 31280, 'Y este es el caso de Karol que mejor dicho no sé cómo presentarte porque siento que']
[31280, 37040, 'esta historia tuya es muy mágica pero además muy especial porque es un caso distinto.']
[37040, 41920, 'Ya ustedes saben que nosotros tenemos una convocatoria abierta y que todas las mujeres']
[41920, 46800, 'que quieren hacer parte de estos episodios se suscriben y hacemos un proceso de curaduría']
[46800, 48520, 'pero con Karol fue diferente.']
[48520, 53640, 'No voy a ser yo quien les cuente sino ella directamente que les cue